# STA497: 5-fold cross-validation, XGBoost vs Random Forest

The original models (`06_STA497_models.ipynb`) were scored on one shuffled 60/20/20 split. This notebook
re-scores the NOx models with 5-fold cross-validation and adds back the Random Forest baseline,
whose original cells were deleted. The RF settings are new (`RF_PARAMS` in `python/sta497/models.py`).

Every experiment runs with both models and under two fold schemes:

- **random**: shuffled k-fold over 1-minute rows. Neighbouring minutes are nearly identical, so the
  test fold has seen almost the same data in training, and the score is optimistic.
- **day**: grouped k-fold by calendar day. A day is never split between training and test.

The gap between the two schemes measures how much temporal leakage inflated the random-split scores.

The code lives in `python/sta497/cv.py`, which `scripts/10_python_kfold_cv.py` also uses. The full run takes
a while, and the script is the faster way to get the CSVs.

**Privacy:** this notebook shows metrics. Strip its outputs before committing (the pre-commit hook enforces this).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Find the repo root (the folder holding config.yml) whether Jupyter starts here or at the root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yml").exists())
sys.path.insert(0, str(ROOT / "python"))

from sta497 import cv, models
from sta497.config import output_path

## 1. Data

Three experiments, with the same rows and XGBoost hyperparameters as scripts 07 and 08:

| Experiment | Predictors |
|---|---|
| `full_nox` | 37 sound measures |
| `selected_nox` | 19 frequencies kept after the correlation > 0.9 grouping |
| `sta465_selected_nox` | 29: 18 frequencies, 4 well-production, 7 met (flare table, one copy per minute) |

In [ ]:
experiments = cv.load_experiments()
{name: (len(data), len(predictors)) for name, (data, predictors, response) in experiments.items()}

## 2. Cross-validation

5 folds per experiment × model × scheme (60 fits in all). XGBoost uses the same fixed 1000 rounds as the
original final models; Random Forest uses `models.RF_PARAMS`.

In [ ]:
rows = []
for name, (data, predictors, response) in experiments.items():
    for model in cv.MODELS:
        for scheme in cv.SCHEMES:
            rows += cv.run(name, data, predictors, response, model, scheme)
            print(f"{name} {model} {scheme}: done", flush=True)
by_fold = pd.DataFrame(rows)

## 3. Results

Mean and SD across the 5 folds. Saved to `outputs/python/sta497_cv/` (gitignored).

In [ ]:
summary = cv.summarize(by_fold)
by_fold.to_csv(output_path("sta497_cv", "metrics_by_fold.csv"), index=False)
summary.to_csv(output_path("sta497_cv", "metrics.csv"), index=False)
summary

In [ ]:
# R² side by side: random vs day-blocked folds, and the leakage gap
r2 = summary.pivot_table(index=["experiment", "model"], columns="scheme", values="test_r2_mean", sort=False)
r2["gap"] = r2["random"] - r2["day"]
r2

## 4. Reading the results

- Quote the **day** scheme as the out-of-sample estimate, since it matches predicting NOx on days the model hasn't seen.
- **XGBoost vs Random Forest:** compare within the same scheme and experiment.
- A large random-minus-day gap means the original random-split R² was driven mostly by autocorrelation.